# 07: RAGAS / Answer-Quality Evaluation

Phase 2. Adds answer-quality evaluation on top of the IR metrics: runs RAGAS and a
correctness check across all four retrieval configs, judged by `gpt-5.5` via the OpenAI
API — independent of both the Llama-3.1-8B generator and the Qwen2.5-14B reference-answer
drafter. Run after notebooks 03 (baseline) and 06 (hybrid).

See `07_ragas_evaluation.md` for the build plan and completion criteria, and
`ragas/07_ragas_evaluation.md` for full technical detail and decisions.

## Cell group 1: Imports and constants

In [2]:
import json
import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

SCRIPTS_DIR = Path("scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

DATA_DIR    = Path("data")
CHROMA_DIR  = DATA_DIR / "chroma_db"
RESULTS_DIR = Path("results")
FIGURES_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)

EVAL_K       = [1, 3, 5, 10]
RRF_K        = 60
GRAPH_HOPS   = 2
TOP_K        = 10

JUDGE_MODEL  = os.getenv("OPENAI_MODEL", "gpt-5.5")
EMBED_MODEL  = "sentence-transformers/all-MiniLM-L6-v2"

ALL_CONFIGS = {
    'dense_only':   'Dense only',
    'sparse_only':  'Sparse only',
    'dense_sparse': 'Dense + Sparse',
    'hybrid':       'Hybrid (+ Graph)',
}

print(f'JUDGE_MODEL: {JUDGE_MODEL}')
print(f'EMBED_MODEL: {EMBED_MODEL}')
print(f'Configs: {list(ALL_CONFIGS)}')

JUDGE_MODEL: gpt-5.5
EMBED_MODEL: sentence-transformers/all-MiniLM-L6-v2
Configs: ['dense_only', 'sparse_only', 'dense_sparse', 'hybrid']


## Quick check: is the OpenAI API reachable?

Minimal connectivity check before building anything on top of it — same idea as
`verify_api_key.py`, condensed into one cell.

In [3]:
from openai import OpenAI, APIError

client = OpenAI()
t0 = time.time()
try:
    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[{"role": "user", "content": "Reply with exactly: OK"}],
    )
    elapsed = time.time() - t0
    reply = resp.choices[0].message.content.strip()
    print(f'OK  {JUDGE_MODEL}  ({elapsed:.2f}s)  reply={reply!r}')
except APIError as e:
    elapsed = time.time() - t0
    print(f'FAILED  ({elapsed:.2f}s)  {type(e).__name__}: {e}')
    raise

OK  gpt-5.5  (4.74s)  reply='OK'


## Cell group 2: Judge and embeddings setup

`ragas==0.4.3` hard-imports a `langchain_community` submodule that no longer exists in
the installed package (confirmed: current latest release, not a version we chose badly —
see `ragas/07_ragas_evaluation.md`). The stub below must run before the first `import
ragas` anywhere in this notebook.

In [4]:
import types

if "langchain_community.chat_models.vertexai" not in sys.modules:
    _stub = types.ModuleType("langchain_community.chat_models.vertexai")
    _stub.ChatVertexAI = type("ChatVertexAI", (), {})
    sys.modules["langchain_community.chat_models.vertexai"] = _stub

import ragas
print(f'ragas {ragas.__version__} imported OK')

C:\Users\tsono\Documents\uoe\disertation\plan\implementation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ragas 0.4.3 imported OK


In [5]:
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings.huggingface_provider import HuggingFaceEmbeddings as RagasHFEmbeddings

# Async client: ragas's collections metrics (Faithfulness, etc.) use ascore()/agenerate()
# internally, which requires an async client. A sync client fails at call time with
# "Cannot use agenerate() with a synchronous client." Jupyter also runs its own event
# loop, so the *sync* score()/generate() path doesn't work here either way - ascore()
# with an async client is the only combination that works inside a notebook cell.
async_client = AsyncOpenAI()
judge = llm_factory(JUDGE_MODEL, client=async_client)

# Second ragas 0.4.3 bug, distinct from the vertexai one: InstructorLLM's reasoning-model
# detection (ragas/llms/base.py: _map_openai_params/is_reasoning_model) does
# int(version_str) on the bit after "gpt-" to decide if a model needs max_completion_tokens
# instead of max_tokens (plus temperature forced to 1.0, top_p removed - all three are
# gated on the same detection). int("5.5") raises ValueError, so "gpt-5.5" silently fails
# the check and is treated as a legacy model - the API then rejects the resulting request
# for using max_tokens (confirmed live, same error verify_api_key.py hit earlier).
# Patch model_args directly to what the correct branch would have produced.
judge.model_args["max_completion_tokens"] = judge.model_args.pop("max_tokens")
judge.model_args["temperature"] = 1.0
judge.model_args.pop("top_p", None)

print(f'judge OK: {type(judge).__name__}  model={JUDGE_MODEL}  is_async={judge.is_async}')
print(f'model_args (patched): {judge.model_args}')

embeddings = RagasHFEmbeddings(model=EMBED_MODEL, use_api=False)
print(f'embeddings OK: {type(embeddings).__name__}  model={EMBED_MODEL}')

judge OK: InstructorLLM  model=gpt-5.5  is_async=True
model_args (patched): {'temperature': 1.0, 'max_completion_tokens': 1024}


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2582.19it/s]

embeddings OK: HuggingFaceEmbeddings  model=sentence-transformers/all-MiniLM-L6-v2


In [6]:
from ragas.metrics.collections import Faithfulness, ContextRecall, ContextPrecisionWithReference, AnswerRelevancy

metric_faithfulness = Faithfulness(llm=judge)
metric_context_recall = ContextRecall(llm=judge)
metric_context_precision = ContextPrecisionWithReference(llm=judge)
metric_answer_relevancy = AnswerRelevancy(llm=judge, embeddings=embeddings)

print('All four metric objects constructed OK:')
for m in [metric_faithfulness, metric_context_recall, metric_context_precision, metric_answer_relevancy]:
    print(f'  {m.name}')

All four metric objects constructed OK:
  faithfulness
  context_recall
  context_precision_with_reference
  answer_relevancy


In [6]:
import inspect

print('=== Faithfulness.score parameters ===')
sig = inspect.signature(metric_faithfulness.score)
print(sig)
print()

# Look for a components/required-fields hint
for attr in ['_required_columns', 'required_columns', 'input_model', 'signature']:
    if hasattr(metric_faithfulness, attr):
        print(f'{attr}:', getattr(metric_faithfulness, attr))

=== Faithfulness.score parameters ===
(**kwargs) -> ragas.metrics.result.MetricResult



In [14]:
# Tiny synthetic smoke test - not real project data, just confirming the call shape works end to end.
result = await metric_faithfulness.ascore(
    user_input="Where was Einstein born?",
    response="Einstein was born in Germany.",
    retrieved_contexts=["Albert Einstein was born in Ulm, in the Kingdom of Wurttemberg in the German Empire, on 14 March 1879."],
)
print(f'type: {type(result).__name__}')
print(f'value: {result.value}')

type: MetricResult
value: 1.0


In [15]:
# Confirm the bulk path too - this is what cell groups 4-5 will actually use for 50-200 rows.
results = await metric_faithfulness.abatch_score([
    dict(
        user_input="Where was Einstein born?",
        response="Einstein was born in Germany.",
        retrieved_contexts=["Albert Einstein was born in Ulm, in the Kingdom of Wurttemberg in the German Empire, on 14 March 1879."],
    ),
    dict(
        user_input="Where was Einstein born?",
        response="Einstein was born in France.",
        retrieved_contexts=["Albert Einstein was born in Ulm, in the Kingdom of Wurttemberg in the German Empire, on 14 March 1879."],
    ),
])
for r in results:
    print(f'{r.value}')

1.0
0.0


## Cell group 3: Load retrievers and test set

Loads the same dense+sparse+graph retrievers as notebooks 05/06 (`load_retrievers()`),
plus `data/test_set.jsonl`, which now carries **both** `reference_14b` and `reference_72b`
(not a single `reference` field — see the 2026-07-05 Change log entry in
`07_ragas_evaluation.md`, since both drafter models are being kept for comparison).

In [7]:
import time
from retriever import load_retrievers, dense_only_retrieve, sparse_only_retrieve, dense_sparse_retrieve, hybrid_retrieve

t0 = time.time()
vectorstore, bm25, clauses, G = load_retrievers(DATA_DIR, CHROMA_DIR)
clause_lookup = {c['clause_id']: c for c in clauses}
print(f'Loaded {len(clauses)} clauses  ({time.time()-t0:.1f}s)')
print(f'Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

TEST_SET_PATH = DATA_DIR / 'test_set.jsonl'
test_set = [
    json.loads(line)
    for line in TEST_SET_PATH.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
print(f'\nLoaded {len(test_set)} test queries')
missing_ref = [q for q in test_set if not q.get('reference_14b') or not q.get('reference_72b')]
print(f'Rows missing reference_14b or reference_72b: {len(missing_ref)}')
print(f'Example row keys: {list(test_set[0].keys())}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3965.61it/s]

Loaded 2568 clauses  (6.0s)
Graph: 2826 nodes, 5299 edges

Loaded 50 test queries
Rows missing reference_14b or reference_72b: 0
Example row keys: ['query', 'gold_ids', 'query_type', 'reference_14b', 'reference_72b']


## Cell group 4: Rebuild top-k contexts for all four configs

Same four retrieval functions as notebook 06's ablation (`dense_only_retrieve`,
`sparse_only_retrieve`, `dense_sparse_retrieve`, `hybrid_retrieve`), run here over all 50
queries so each config's retrieved contexts are available for both answer generation and
retrieval-side RAGAS. All local, no GPU/API cost.

In [8]:
def retrieve_for_config(config: str, query: str) -> list:
    if config == 'dense_only':
        return dense_only_retrieve(vectorstore, clauses, query, k=TOP_K)
    if config == 'sparse_only':
        return sparse_only_retrieve(bm25, clauses, query, k=TOP_K)
    if config == 'dense_sparse':
        return dense_sparse_retrieve(query, vectorstore, bm25, clauses, k=TOP_K, rrf_k=RRF_K)
    if config == 'hybrid':
        return hybrid_retrieve(query, vectorstore, bm25, clauses, G, k=TOP_K, graph_hops=GRAPH_HOPS, rrf_k=RRF_K)
    raise ValueError(f'unknown config: {config}')

t0 = time.time()
retrieval_results = {}
for config in ALL_CONFIGS:
    rows = []
    for item in test_set:
        retrieved = retrieve_for_config(config, item['query'])
        rows.append({**item, 'retrieved': retrieved})
    retrieval_results[config] = rows
    n_empty = sum(1 for r in rows if not r['retrieved'])
    print(f'{config:<14} {len(rows)} queries retrieved  (top-1 empty: {n_empty})')

print(f'\nTotal retrieval time: {time.time()-t0:.1f}s')

dense_only     50 queries retrieved  (top-1 empty: 0)


sparse_only    50 queries retrieved  (top-1 empty: 0)


dense_sparse   50 queries retrieved  (top-1 empty: 0)


hybrid         50 queries retrieved  (top-1 empty: 0)

Total retrieval time: 21.6s


## Cell group 5: Load the generator

Same Llama-3.1-8B-Instruct setup as notebook 02 (NF4 4-bit, local 6GB GPU) — this
notebook's `RESULTS_DIR`/`GRAPH_HOPS`/`RRF_K` constants are already defined in cell group
1, so only the model-specific pieces are added here. `format_context()` truncates each
clause to 800 chars, matching notebook 02, to keep prompts within the 8B model's context
comfortably across all four configs' retrieved sets.

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from transformers import pipeline as hf_pipeline_fn
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

MODEL_ID = 'NousResearch/Meta-Llama-3.1-8B-Instruct'  # fixed regardless of hardware - the four
                                                       # retrieval configs are compared against the
                                                       # same generator; see plan.md Sec 5,
                                                       # "Generator model identity"
LOAD_IN_4BIT = True   # True = NF4 4-bit for 6GB local GPU; False = BF16 for H200 cluster (same model)
MAX_NEW_TOKENS = 1024

SYSTEM_PROMPT = '''You are an AML compliance analyst. Answer using ONLY the regulatory clauses below.

Rules:
1. Every factual claim must be followed by [clause_id] inline.
2. State explicitly if the provided context does not answer the question.
3. Output valid JSON with two keys:
   - answer: your response with inline [clause_id] citations
   - citations: list of clause_id strings you cited

Context:
{context}'''

def format_context(results: list) -> str:
    parts = []
    for r in results:
        cid  = r['clause_id']
        text = r['text'][:800]
        parts.append(f'[{cid}]\n{text}')
    return '\n\n---\n\n'.join(parts)

prompt = ChatPromptTemplate.from_messages([
    ('system', SYSTEM_PROMPT),
    ('human', '{query}'),
])

_hf_token = os.getenv('HF_TOKEN', '')
if not _hf_token:
    raise EnvironmentError('HF_TOKEN not set -- add it to .env and restart the kernel.')
if not torch.cuda.is_available():
    raise EnvironmentError('No CUDA device found -- this notebook requires a GPU.')

torch.cuda.empty_cache()
if LOAD_IN_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
    )
    model_kwargs = {'quantization_config': quant_config, 'device_map': {'': 0}}
else:
    model_kwargs = {'torch_dtype': torch.bfloat16, 'device_map': 'auto'}

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
gen_pipe = hf_pipeline_fn(
    'text-generation', model=model, tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS, do_sample=False, return_full_text=False,
)
llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipe))
parser = JsonOutputParser()
chain = prompt | llm | parser

precision = 'NF4 4-bit' if LOAD_IN_4BIT else 'BF16'
print(f'Model loaded: {MODEL_ID}  ({time.time()-t0:.1f}s)')
print(f'Precision: {precision}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM free/total (GB): {[x // 1024**3 for x in torch.cuda.mem_get_info()]}')

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/291 [00:04<19:50,  4.11s/it]

Loading weights:   1%|          | 2/291 [00:06<14:12,  2.95s/it]

Loading weights:   1%|▏         | 4/291 [00:07<06:27,  1.35s/it]

Loading weights:   2%|▏         | 5/291 [00:07<04:48,  1.01s/it]

...

Loading weights:  90%|█████████ | 263/291 [00:39<00:03,  8.50it/s]

Loading weights:  91%|█████████ | 265/291 [00:39<00:03,  7.44it/s]

Loading weights:  92%|█████████▏| 267/291 [00:40<00:05,  4.74it/s]

Loading weights:  93%|█████████▎| 270/291 [00:40<00:03,  6.50it/s]

Loading weights:  93%|█████████▎| 272/291 [00:40<00:02,  7.47it/s]

In [ ]:
# Single-query timing smoke test before committing to the full 200-query loop below -
# 6GB laptop GPU, so worth confirming per-query latency first rather than assuming.
test_query = test_set[0]['query']
test_retrieved = retrieve_for_config('dense_only', test_query)
test_context = format_context(test_retrieved)
print(f'Test query: {test_query!r}')
print(f'Context length (chars): {len(test_context)}')

t0 = time.time()
result = chain.invoke({'query': test_query, 'context': test_context})
elapsed = time.time() - t0
print(f'\nGenerated in {elapsed:.1f}s')
print(result)
print(f'\nEstimated total for 200 generations: {elapsed * 200 / 60:.1f} min (assuming similar per-query cost)')

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


[transformers] Both `max_new_tokens` (=1024) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Test query: 'Under MLR 2017 regulation 4, at what point is an estate agent considered to have entered into a business relationship with a purchaser?'
Context length (chars): 6187


## Cell group 6: Generate 200 answers

The single-query timing test above (129.6s) implies ~7+ hours for all 200 generations
(4 configs x 50 queries) on this local 6GB GPU — not run in full here. The function below
matches `generation_cluster/generate_answers.py` exactly (same model, same prompt, same
config loop), which was run on the H200 cluster instead (BF16 instead of NF4 4-bit — same
`Llama-3.1-8B-Instruct` generator either way, see plan.md Sec 5, "Generator model
identity"). Included here for a complete, reviewable picture of the methodology, not to
be executed against the full test set locally.

In [15]:
def generate_all_answers() -> list[dict]:
    """Generate answers for all (config, query) pairs; write results/answers.jsonl incrementally.

    Matches generation_cluster/generate_answers.py exactly - see that file's docstring
    for why the generator model is fixed regardless of where this runs.
    """
    rows = []
    t0 = time.time()
    n_total = len(ALL_CONFIGS) * len(test_set)
    n_done = 0
    n_errors = 0
    output_path = RESULTS_DIR / 'answers.jsonl'
    with output_path.open('w', encoding='utf-8') as f:
        for config in ALL_CONFIGS:
            for item in test_set:
                query = item['query']
                retrieved = retrieve_for_config(config, query)
                context = format_context(retrieved)
                try:
                    parsed = chain.invoke({'query': query, 'context': context})
                    answer = parsed.get('answer', '')
                    citations = parsed.get('citations', [])
                except Exception as exc:
                    answer = f'Generation error: {exc}'
                    citations = []
                    n_errors += 1

                row = {
                    'query': query,
                    'gold_ids': item['gold_ids'],
                    'query_type': item.get('query_type', 'unknown'),
                    'config': config,
                    'retrieved': [r['clause_id'] for r in retrieved],
                    'answer': answer,
                    'citations': citations,
                }
                rows.append(row)
                f.write(json.dumps(row, ensure_ascii=False) + '\n')
                f.flush()

                n_done += 1
                elapsed = time.time() - t0
                avg = elapsed / n_done
                remaining = (n_total - n_done) * avg
                logger.info(
                    '[%d/%d] config=%s  %r -> %d chars  (avg %.1fs/query, ~%.0fs remaining)',
                    n_done, n_total, config, query[:50], len(answer), avg, remaining,
                )

    logger.info('Done: %d rows written to %s  (%d generation errors)', len(rows), output_path, n_errors)
    return rows

print('generate_all_answers() defined - not called here (would take ~7 hours locally).')
print('Actual run: generation_cluster/generate_answers.py on the H200 cluster.')

generate_all_answers() defined - not called here (would take ~7 hours locally).
Actual run: generation_cluster/generate_answers.py on the H200 cluster.


In [16]:
ANSWERS_PATH = RESULTS_DIR / 'answers.jsonl'

if ANSWERS_PATH.exists():
    answers = [
        json.loads(line)
        for line in ANSWERS_PATH.read_text(encoding='utf-8').splitlines()
        if line.strip()
    ]
    logger.info('Loaded %d generated answers from %s', len(answers), ANSWERS_PATH)
    print(f'Loaded {len(answers)} answers  (expected {len(ALL_CONFIGS) * len(test_set)})')
    by_config = {c: sum(1 for a in answers if a['config'] == c) for c in ALL_CONFIGS}
    print(f'Per config: {by_config}')
    n_errors = sum(1 for a in answers if a['answer'].startswith('Generation error:'))
    print(f'Generation errors: {n_errors}')
else:
    answers = []
    print(f'{ANSWERS_PATH} not found yet.')
    print('Run generation_cluster/generate_answers.py on the cluster, then copy')
    print(f'results/answers.jsonl back to {RESULTS_DIR.resolve()} before continuing.')

results\answers.jsonl not found yet.
Run generation_cluster/generate_answers.py on the cluster, then copy
results/answers.jsonl back to C:\Users\tsono\Documents\uoe\disertation\plan\implementation\results before continuing.


### Free the generator

The remaining cell groups (retrieval-side and answer-side RAGAS) only need the `gpt-5.5`
judge and the local embeddings model — not the Llama generator. Freeing it here matches
the original pipeline design (`07_ragas_evaluation.md` Sec Pipeline, step 3) and avoids
holding ~5GB of VRAM for the rest of the notebook for no reason.

In [17]:
import gc

del model, tokenizer, gen_pipe, llm, chain, parser, prompt
gc.collect()
torch.cuda.empty_cache()

print('Generator freed.')
print(f'VRAM free/total (GB): {[x // 1024**3 for x in torch.cuda.mem_get_info()]}')

Generator freed.
VRAM free/total (GB): [0, 5]
